# Gowalla — recovering LLMGPR's Table 1 column

Same exercise as the Foursquare recovery (`LLMGPR_TRACK.md` §1), for the Gowalla column of
LLMGPR (CIKM'25) Table 1:

| users | groups | POIs | cats | user ck | group ck | ck/user | ck/group | users/group |
|---|---|---|---|---|---|---|---|---|
| 31,751 | 2,186 | 81,123 | 537 | 862,502 | 7,738 | 27.16 | 3.54 | 2.95 |

**The dump.** LLMGPR cites its Gowalla as [24] = Liu, Liu, Aberer, Miao (CIKM'13), whose data is
the full Gowalla crawl once hosted at `yongliu.org/datasets` — 36,001,959 check-ins / 319,063
users / 2,844,076 spots **with per-spot categories** (the SNAP `loc-gowalla` release has none) and
a friendship graph. Mirror: figshare article **22126586** (md5s verified below). The paper says
all three datasets cover New York, Los Angeles and Chicago; the same Yang city centres and
bounding boxes as the Foursquare track are used here.

**Verdict, in brief** (details in `LLMGPR_GOWALLA.md`): three of the four columns are
reproducible to ≤ 1.2% (best sweep row 0.995 on users/POIs/check-ins), but — exactly as on
Foursquare — only under readings their §4.1 does not describe: **no ≥10 user cut** (the
documented rule tops out at match 0.74 and keeps 0.37× their users), `#POIs` as a **~28 km
catalogue**, and `#categories` **unrecoverable from this dump** (≤ 344 in-region vs their 537).
Adopted build: R = 15 km collection, day-level dedup, no user filter, km ≤ 28 catalogue →
**users 0.997× / POIs 1.001× / check-ins 1.047×**.

In [1]:
import os, gc, itertools, math, re, subprocess
import numpy as np, pandas as pd

REPO = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
RAW  = os.path.join(REPO, "data", "gowalla_raw")
OUT  = os.path.join(REPO, "data", "gowalla")
os.makedirs(RAW, exist_ok=True); os.makedirs(OUT, exist_ok=True)
CHUNK = 2_000_000
TARGET = dict(users=31_751, pois=81_123, cats=537, ck=862_502)

# Yang's city centres and the confirmed boxes -- identical to the Foursquare track
CITY_CENTRE = {"New York": (40.707864, -73.905237), "Chicago": (41.826546, -87.641298),
               "Los Angeles": (34.000002, -118.250001)}
BBOX = {"New York": (-74.3, -73.6, 40.4, 41.0), "Chicago": (-88.0, -87.5, 41.6, 42.1),
        "Los Angeles": (-118.7, -117.6, 33.6, 34.4)}
PADS = (0.0, 0.1, 0.2, 0.3, 0.4); PAD_MAX = max(PADS)

# figshare 22126586 -- the yongliu.org Gowalla crawl (Liu et al. CIKM'13)
FILES = {  # name -> (figshare file id, md5)
    "gowalla_category_structure.json": (39334343, "236c90adf2c0bda63c318404ff19b08f"),
    "gowalla_checkins.csv":            (39334346, "f61b7b835dc807562381bca206c1d7b6"),
    "gowalla_friendship.csv":          (39334349, "71b5b9121bd1fed48b6947327b199590"),
    "gowalla_spots_subset1.csv":       (39334352, "9023df52ab0ddb9011daeff9fec49698"),
    "gowalla_spots_subset2.csv":       (39334355, "3e81e6e18afbc8c973c2afee26270c9f"),
    "gowalla_userinfo.csv":            (39334358, "7720d985829eb3c27dde9eaa4f605473"),
}
import hashlib
def md5(fp, buf=1 << 22):
    h = hashlib.md5()
    with open(fp, "rb") as f:
        while chunk := f.read(buf): h.update(chunk)
    return h.hexdigest()
for name, (fid, want) in FILES.items():
    fp = os.path.join(RAW, name)
    if not os.path.exists(fp):
        print("fetching", name, flush=True)
        url = f"https://ndownloader.figshare.com/files/{fid}"
        assert subprocess.run(f'curl -sL --fail --retry 3 -o "{fp}" "{url}"', shell=True).returncode == 0
        assert md5(fp) == want, f"md5 mismatch on {name}"
print("all six files present")

all six files present


## 1. The 3-city spot universe — pad levels, distance to centre, categories

In [2]:
UNIV = os.path.join(RAW, "_universe_pad04.parquet")
CAT_ID = re.compile(r"/categories/(\d+)"); CAT_NAME = re.compile(r"'name':\s*'([^']*)'")

def in_padded_boxes(df):
    keep = pd.Series(False, index=df.index); cty = pd.Series("?", index=df.index)
    for c, (lo1, lo2, la1, la2) in BBOX.items():
        m = (df["lng"].between(lo1 - PAD_MAX, lo2 + PAD_MAX)
             & df["lat"].between(la1 - PAD_MAX, la2 + PAD_MAX))
        keep |= m; cty[m & (cty == "?")] = c
    return keep, cty

if not os.path.exists(UNIV):
    parts = []
    for fn, enc, cats in (("gowalla_spots_subset1.csv", "utf-8", True),
                          ("gowalla_spots_subset2.csv", "latin-1", False)):
        usecols = ["id", "lng", "lat"] + (["spot_categories"] if cats else [])
        for chx in pd.read_csv(os.path.join(RAW, fn), chunksize=500_000, usecols=usecols,
                               dtype={"id": np.int64}, encoding=enc, on_bad_lines="skip"):
            chx["lat"] = pd.to_numeric(chx["lat"], errors="coerce")
            chx["lng"] = pd.to_numeric(chx["lng"], errors="coerce")
            chx = chx.dropna(subset=["lat", "lng"])
            keep, cty = in_padded_boxes(chx)
            if not keep.any(): continue
            sub = chx[keep].copy(); sub["city"] = cty[keep]
            sc = sub.pop("spot_categories").fillna("") if cats else pd.Series("", index=sub.index)
            sub["cat_ids"] = sc.map(lambda s: ",".join(CAT_ID.findall(s)))
            sub["cat_names"] = sc.map(lambda s: "|".join(CAT_NAME.findall(s)))
            parts.append(sub)
    pois = pd.concat(parts, ignore_index=True).drop_duplicates("id").reset_index(drop=True)
    del parts; gc.collect()

    lat, lon = pois["lat"].to_numpy(), pois["lng"].to_numpy()
    pad_level = np.full(len(pois), np.inf)
    for c, (lo1, lo2, la1, la2) in BBOX.items():
        m = (pois["city"] == c).to_numpy()
        for p in sorted(PADS):
            inside = m & (lon >= lo1 - p) & (lon <= lo2 + p) & (lat >= la1 - p) & (lat <= la2 + p)
            pad_level = np.where(inside & (pad_level == np.inf), p, pad_level)
    pois["pad_level"] = pad_level

    def haversine_km(la, lo, la0, lo0):
        R = 6371.0088
        p, p0 = np.radians(la), math.radians(la0)
        dp, dl = p - p0, np.radians(lo - lo0)
        return 2 * R * np.arcsin(np.sqrt(np.sin(dp/2)**2 + np.cos(p)*math.cos(p0)*np.sin(dl/2)**2))
    km = np.full(len(pois), np.inf)
    for c, (la0, lo0) in CITY_CENTRE.items():
        m = (pois["city"] == c).to_numpy()
        if m.any(): km[m] = haversine_km(lat[m], lon[m], la0, lo0)
    pois["km"] = km
    pois.to_parquet(UNIV, index=False)
pois = pd.read_parquet(UNIV)
print(f"universe at pad {PAD_MAX}: {len(pois):,} spots | "
      f"{(pois.cat_ids != '').mean()*100:.1f}% with a category")
print(pois[pois.pad_level == 0]["city"].value_counts().to_string())

universe at pad 0.4: 151,642 spots | 96.8% with a category
city
Los Angeles    55783
New York       36440
Chicago        17468


## 2. Check-ins — full-file scan, global per-user counts, 3-city rows

In [3]:
CKP, GTP = os.path.join(RAW, "_checkins_pad04.parquet"), os.path.join(RAW, "_user_global_counts.parquet")
if not (os.path.exists(CKP) and os.path.exists(GTP)):
    KEEP = set(pois["id"].to_numpy())
    parts, gtot, seen = [], None, 0
    tmin, tmax = "9999", "0000"
    for chx in pd.read_csv(os.path.join(RAW, "gowalla_checkins.csv"), chunksize=CHUNK,
                           dtype={"userid": np.int64, "placeid": np.int64, "datetime": str},
                           on_bad_lines="skip"):
        seen += len(chx)
        vc = chx["userid"].value_counts()
        gtot = vc if gtot is None else gtot.add(vc, fill_value=0)
        dtc = chx["datetime"].dropna()
        if len(dtc): tmin, tmax = min(tmin, dtc.min()), max(tmax, dtc.max())
        parts.append(chx[chx["placeid"].isin(KEEP)])
        print(f"\rscanned {seen:,}", end="", flush=True)
    ck = pd.concat(parts, ignore_index=True); del parts; gc.collect()
    print(f"\nwhole file: {seen:,} ck / {len(gtot):,} users ({seen/len(gtot):.1f} each) | {tmin} .. {tmax}")
    ck.to_parquet(CKP, index=False)
    gtot.astype("int64").rename_axis("userid").rename("n_global").to_frame().reset_index().to_parquet(GTP, index=False)
ck = pd.read_parquet(CKP)
gt = pd.read_parquet(GTP).set_index("userid")["n_global"]

vpos = pd.Series(np.arange(len(pois)), index=pois["id"])
vid = ck["placeid"].map(vpos).to_numpy().astype(np.int64)
uid, users_u = pd.factorize(ck["userid"], sort=False)
NU, NV = len(users_u), len(pois)
vkm, vpad = pois["km"].to_numpy(), pois["pad_level"].to_numpy()
vcat1 = pd.factorize(pois["cat_ids"].str.split(",").str[0].replace("", np.nan),
                     use_na_sentinel=True)[0]
g_user = pd.Series(users_u).map(gt).fillna(0).to_numpy().astype("int64")
month = ck["datetime"].str[:7].to_numpy()
day = ck["datetime"].str.slice(0, 10)
dupmask = ~pd.DataFrame({"u": ck["userid"], "p": ck["placeid"], "d": day}).duplicated().to_numpy()
pad0 = (vpad <= 0)[vid]
print(f"pad-0.4 rows {len(ck):,} | pad-0 boxes: {int(pad0.sum()):,} ck / "
      f"{len(np.unique(uid[pad0])):,} users / {len(np.unique(vid[pad0])):,} spots visited")
print(f"TARGET: {TARGET['ck']:,} ck / {TARGET['users']:,} users / {TARGET['pois']:,} POIs / {TARGET['cats']} cats")

pad-0.4 rows 2,419,169 | pad-0 boxes: 2,003,626 ck / 42,514 users / 109,612 spots visited
TARGET: 862,502 ck / 31,751 users / 81,123 POIs / 537 cats


## 3. Eliminations — what their column can and cannot be

Three measurements kill the obvious readings before any sweep:

1. **The documented rule** (*"users and POIs with less than 10 interactions are removed"*) keeps
   ~18.6k users in the plain boxes and ~11.6k in a 15 km region — 0.37–0.59× their 31,751 — and
   every activity cut *raises* mean ck/user above the unfiltered 47.1, while theirs is **27.16**.
   No region or window rescues it: best match over the whole grid is **0.74**.
2. **Whole-history accounting** (the reading that recovered Foursquare at 0.978) explodes here:
   retained users' worldwide venues number ~1.2M (15× their #POIs) at ~490 ck/user.
3. **#categories = 537 is unreachable from this dump in-region**: distinct first category ids max
   at 341 (pad 0.4), distinct raw `spot_categories` strings at 344, per-city-summed vocabularies
   start at 739, the taxonomy file holds 266 names, the global crawl vocabulary is 667, and
   whole-history visited sets give 637–667. Nothing lands near 537.

In [4]:
# the numbers behind the three eliminations
def show_kill(label, users, ckn):
    print(f"{label:>42}: {users:>7,} users ({users/TARGET['users']:.2f}x) | "
          f"{ckn:>10,} ck ({ckn/TARGET['ck']:.2f}x, {ckn/max(users,1):.1f}/u)")

raw_pu0 = np.bincount(uid[pad0], minlength=NU)
keep10 = raw_pu0 >= 10
rows10 = pad0 & keep10[uid]
rows10 &= (np.bincount(vid[rows10], minlength=NV) >= 10)[vid]
show_kill("documented >=10/>=10, plain boxes, in/in", int((np.bincount(uid[rows10], minlength=NU) > 0).sum()), int(rows10.sum()))

m15 = (vkm <= 15)[vid]
pu15 = np.bincount(uid[m15], minlength=NU)
k15 = pu15 >= 10
r15 = m15 & k15[uid]; r15 &= (np.bincount(vid[r15], minlength=NV) >= 10)[vid]
show_kill("documented >=10/>=10, R=15 km", int((np.bincount(uid[r15], minlength=NU) > 0).sum()), int(r15.sum()))
show_kill("unfiltered, plain boxes", len(np.unique(uid[pad0])), int(pad0.sum()))
show_kill("unfiltered, R=15 km", len(np.unique(uid[m15])), int(m15.sum()))

print("\ncategory bases (theirs: 537)")
for lab, mask in (("pad 0 boxes", vpad <= 0), ("pad 0.4", vpad <= PAD_MAX), ("km<=28", vkm <= 28)):
    n1 = int((np.unique(vcat1[mask]) >= 0).sum())
    nsum = sum(int((np.unique(vcat1[mask & (pois['city'] == c).to_numpy()]) >= 0).sum())
               for c in BBOX)
    raws = pois.loc[mask & (pois.cat_ids != "").to_numpy(), "cat_ids"].nunique()
    print(f"  {lab:>12}: distinct ids {n1} | distinct id-strings {raws} | per-city sum {nsum}")
struct = open(os.path.join(RAW, "gowalla_category_structure.json")).read()
print(f"  taxonomy file: {len(set(re.findall(r'/categories/([0-9]+)', struct)))} distinct ids | "
      f"global crawl vocabulary: 667 (measured over all 2.72M subset1 spots)")

  documented >=10/>=10, plain boxes, in/in:  18,606 users (0.59x) |  1,658,817 ck (1.92x, 89.2/u)
             documented >=10/>=10, R=15 km:  11,627 users (0.37x) |    757,936 ck (0.88x, 65.2/u)
                   unfiltered, plain boxes:  42,514 users (1.34x) |  2,003,626 ck (2.32x, 47.1/u)
                       unfiltered, R=15 km:  31,667 users (1.00x) |    934,048 ck (1.08x, 29.5/u)

category bases (theirs: 537)
   pad 0 boxes: distinct ids 335 | distinct id-strings 335 | per-city sum 946
       pad 0.4: distinct ids 341 | distinct id-strings 341 | per-city sum 962
        km<=28: distinct ids 326 | distinct id-strings 326 | per-city sum 930
  taxonomy file: 266 distinct ids | global crawl vocabulary: 667 (measured over all 2.72M subset1 spots)


## 4. The sweep

Axes, mirroring `llmgpr-boundaries-and-region.ipynb` but reshaped by the eliminations above:
collection region (km radius or padded boxes) × end-of-window × day-dedup × user threshold
(1 = none, 3, 5, 10) × threshold basis (in-region raw / in-region distinct-POIs / global) ×
check-in basis (raw / distinct pairs) × per-user cap (none / 200 / 500) × `#POIs` reading
(visited, or a km catalogue). `match4` scores all four columns; `match3` drops the
provably-unreachable `#categories`.

In [5]:
def match(g, keys, t=TARGET):
    return float(np.mean([min(g[k], t[k]) / max(g[k], t[k]) for k in keys]))

REGIONS = [("km", r) for r in (13, 14, 15, 16, 17, 20, 25)] + [("box", 0.0), ("box", 0.4)]
WINDOWS = ("full", "2011-06", "2011-04", "2010-10")
CATALOG = {f"km{r}": (vkm <= r) for r in (15, 20, 25, 27, 28, 30)}
CATALOG["box0"] = vpad <= 0
cat_stats = {k: (int(m.sum()), int((np.unique(vcat1[m]) >= 0).sum())) for k, m in CATALOG.items()}

rows_out = []
for (rk, rv), win, ded in itertools.product(REGIONS, WINDOWS, (False, True)):
    reg_v = (vkm <= rv) if rk == "km" else (vpad <= rv)
    base = reg_v[vid]
    if win != "full": base = base & (month <= win)
    if ded: base = base & dupmask
    if not base.any(): continue
    raw_pu_all = np.bincount(uid[base], minlength=NU)
    key = np.unique(uid[base].astype(np.int64) * NV + vid[base])
    pair_pu_all = np.bincount((key // NV).astype(np.int64), minlength=NU)
    for Tu, tb in itertools.product((1, 3, 5, 10), ("in", "pairs", "global")):
        cnt = raw_pu_all if tb == "in" else pair_pu_all if tb == "pairs" else g_user
        keep_u = (cnt >= Tu) & (raw_pu_all > 0)
        n_users = int(keep_u.sum())
        if n_users == 0: continue
        rows = base & keep_u[uid]
        visited = np.bincount(vid[rows], minlength=NV) > 0
        vis = (int(visited.sum()), int((np.unique(vcat1[visited]) >= 0).sum()))
        raw_pu, pair_pu = raw_pu_all[keep_u], pair_pu_all[keep_u]
        for cb, cap in itertools.product(("raw", "pairs"), (None, 200, 500)):
            per = raw_pu if cb == "raw" else pair_pu
            if cap: per = np.minimum(per, cap)
            nck = int(per.sum())
            for pn, (npo, nc) in [("visited", vis)] + list(cat_stats.items()):
                g = dict(users=n_users, pois=npo, cats=nc, ck=nck)
                m4, m3 = match(g, tuple(TARGET)), match(g, ("users", "pois", "ck"))
                if m3 >= 0.80 or (Tu == 10 and tb == "in" and cb == "raw" and cap is None):
                    rows_out.append((m4, m3, f"{rk}{rv}", win, ded, Tu, tb, cb, cap, pn, g))
print(f"{len(rows_out):,} rows kept")

def show(rs, title, keyi, n=12):
    rs = sorted(rs, key=lambda r: -r[keyi])
    print(f"\n### {title}")
    hdr = (f"{'m4':>6}{'m3':>6}{'region':>8}{'win':>9}{'ded':>4}{'Tu':>3}{'tb':>7}{'cb':>6}"
           f"{'cap':>5}{'POIs-as':>9}{'users':>8}{'POIs':>9}{'cats':>5}{'ck':>10}{'ck/u':>6}")
    print(hdr); print("-" * len(hdr))
    for m4, m3, reg, win, ded, Tu, tb, cb, cap, pn, g in rs[:n]:
        print(f"{m4:>6.3f}{m3:>6.3f}{reg:>8}{win:>9}{str(ded)[0]:>4}{Tu:>3}{tb:>7}{cb:>6}"
              f"{str(cap or '-'):>5}{pn:>9}{g['users']:>8,}{g['pois']:>9,}{g['cats']:>5}"
              f"{g['ck']:>10,}{g['ck']/g['users']:>6.1f}")
    print("-" * len(hdr))
    print(f"{'TGT':>6}{'':>57}{TARGET['users']:>8,}{TARGET['pois']:>9,}{TARGET['cats']:>5}"
          f"{TARGET['ck']:>10,}{TARGET['ck']/TARGET['users']:>6.1f}")

lit = [r for r in rows_out if r[5] == 10 and r[6] == "in" and r[7] == "raw" and r[8] is None]
show(lit, "documented rule: >=10 in-region, raw counts", 0)
show(rows_out, "best by match4", 0)
show(rows_out, "best by match3 (categories excluded)", 1)

14,286 rows kept

### documented rule: >=10 in-region, raw counts
    m4    m3  region      win ded Tu     tb    cb  cap  POIs-as   users     POIs cats        ck  ck/u
-----------------------------------------------------------------------------------------------------
 0.742 0.787    km15  2011-06   F 10     in   raw    -     km28  11,620   81,240  326   866,350  74.6
 0.741 0.786    km15     full   F 10     in   raw    -     km28  11,628   81,240  326   868,074  74.7
 0.738 0.781    km16  2011-06   T 10     in   raw    -     km28  12,210   81,240  326   898,318  73.6
 0.737 0.781    km16     full   T 10     in   raw    -     km28  12,221   81,240  326   900,202  73.7
 0.737 0.780    km17  2011-04   T 10     in   raw    -     km28  11,368   81,240  326   848,509  74.6
 0.736 0.779    km17  2011-04   F 10     in   raw    -     km28  11,510   81,240  326   883,265  76.7
 0.736 0.779    km15  2011-06   F 10     in   raw    -     km27  11,620   79,100  326   866,350  74.6
 0.735 0.778    

## 5. The recovered reading, and the adopted build

The grid's best row (0.995 on users/POIs/check-ins) needs three unstated knobs — R = 20 km,
a 2011-04 cutoff and a 500-cap — so, per the per-city-radius lesson from the Foursquare track,
it is reported as a fit, not adopted. The adopted build spends **one** fitted parameter:

- **Collection: R = 15 km** around Yang's city centres, full window, **no user filter**,
  day-level dedup (`user, place, day` — standard crawl hygiene, removes 3.3%):
  **31,667 users (0.997×) / 903,045 check-ins (1.047×)**, 28.5 ck/user vs their 27.16.
- **`#POIs` = the km ≤ 28 catalogue: 81,240 (1.001×)** — the region enters only this column,
  exactly like Foursquare's 10 km catalogue. (81,123 sits between km-27's 79,100 and km-28's
  81,240; 28 is kept rather than tuned to 3 decimals.)
- **`#categories`: 326 (0.61×), unrecoverable** — carried as a residual, like Foursquare's
  `#users`, with the §3-style bound: no venue subset of this dump shows 537 in-region.

The residual 4.7% on check-ins closes exactly under any one of the three unstated knobs above;
we refuse to fit them. The `>=10/>=10` core is reported alongside for the fidelity arm.

In [6]:
R_COLLECT, R_CATALOGUE = 15, 28
pois2 = pois.assign(cat1_id=pois["cat_ids"].str.split(",").str[0],
                    cat1_name=pois["cat_names"].str.split("|").str[0])
vk = pois2.set_index("id")

inreg = ck["placeid"].isin(set(pois2.loc[pois2.km <= R_COLLECT, "id"]))
sub = ck[inreg].copy()
raw_pu = sub.groupby("userid").size()
sub["day"] = sub["datetime"].str[:10]
sub = (sub.sort_values("datetime").drop_duplicates(["userid", "placeid", "day"])
          .drop(columns="day").sort_index())
for col, src in (("city", "city"), ("category_id", "cat1_id"), ("category", "cat1_name")):
    sub[col] = sub["placeid"].map(vk[src])
sub["km"] = sub["placeid"].map(vk["km"]).round(3)
sub = sub.rename(columns={"datetime": "utc_time"})

cat = pois2.loc[pois2.km <= R_CATALOGUE,
                ["id", "lat", "lng", "cat1_id", "cat1_name", "city", "km"]].copy()
cat = cat.rename(columns={"id": "venue_id", "cat1_id": "category_id", "cat1_name": "category"})
cat["km"] = cat["km"].round(3)

ded_pu = sub.groupby("userid").size()
users = pd.DataFrame({"userid": ded_pu.index, "n_in_region": ded_pu.to_numpy(),
                      "n_in_region_raw": raw_pu.reindex(ded_pu.index).to_numpy(),
                      "n_global": gt.reindex(ded_pu.index).fillna(0).astype(int).to_numpy()})

fr = pd.read_csv(os.path.join(RAW, "gowalla_friendship.csv"), dtype=np.int64)
us = set(users["userid"].tolist())
fr = fr[fr["userid1"].isin(us) & fr["userid2"].isin(us)].copy()
fr["source"] = "crawl"   # ONE snapshot -- Gowalla has no old/new friendship split

n_cats = cat.loc[cat["category_id"].notna() & (cat["category_id"] != ""), "category_id"].nunique()
stats = dict(users=len(users), pois=len(cat), cats=int(n_cats), ck=len(sub))
m4 = match(stats, tuple(TARGET)); m3 = match(stats, ("users", "pois", "ck"))
print(f"{'column':<24}{'ours':>12}{'theirs':>12}{'ratio':>9}")
print("-" * 57)
for k in ("users", "pois", "cats", "ck"):
    print(f"{k:<24}{stats[k]:>12,}{TARGET[k]:>12,}{stats[k]/TARGET[k]:>9.3f}x")
print(f"{'ck/user':<24}{stats['ck']/stats['users']:>12.2f}{27.16:>12.2f}")
print(f"{'match4 / match3':<24}{m4:>12.3f}{m3:>12.3f}")
print(f"\nleave-one-out feasibility: >=3 in-region ck: {int((ded_pu>=3).sum()):,} users "
      f"({(ded_pu>=3).mean()*100:.0f}%) | friendship: {len(fr):,} directed edges over "
      f"{fr['userid1'].nunique():,} users ({fr['userid1'].nunique()/len(users)*100:.0f}%), "
      f"mean degree {len(fr)/fr['userid1'].nunique():.1f}")

rows = [dict(column=k, ours=stats[k], theirs=TARGET[k], ratio=round(stats[k]/TARGET[k], 4))
        for k in ("users", "pois", "cats", "ck")]
rows += [dict(column="check_ins_per_user", ours=round(stats["ck"]/stats["users"], 2), theirs=27.16,
              ratio=round(stats["ck"]/stats["users"]/27.16, 4)),
         dict(column="match4", ours=round(m4, 4), theirs=1.0, ratio=round(m4, 4)),
         dict(column="match3_users_pois_ck", ours=round(m3, 4), theirs=1.0, ratio=round(m3, 4)),
         dict(column="friend_directed_edges", ours=len(fr), theirs=None, ratio=None),
         dict(column="users_with_friend_edge", ours=fr["userid1"].nunique(), theirs=None, ratio=None)]

for df, stem, comp in ((sub[["userid","placeid","utc_time","city","category_id","category","km"]],
                        "gowalla_final_checkins.csv.gz", "gzip"),
                       (cat, "gowalla_final_catalogue.csv", None),
                       (users, "gowalla_final_users.csv", None),
                       (fr, "gowalla_final_friendships.csv", None),
                       (pd.DataFrame(rows), "gowalla_final_stats.csv", None)):
    fp = os.path.join(OUT, stem)
    df.to_csv(fp, index=False, compression=comp)
    print(f"wrote {stem}  ({len(df):,} rows, {os.path.getsize(fp)/1024**2:.1f} MB)")

column                          ours      theirs    ratio
---------------------------------------------------------
users                         31,667      31,751    0.997x
pois                          81,240      81,123    1.001x
cats                             326         537    0.607x
ck                           903,045     862,502    1.047x
ck/user                        28.52       27.16
match4 / match3                0.890       0.984

leave-one-out feasibility: >=3 in-region ck: 21,632 users (68%) | friendship: 235,898 directed edges over 26,779 users (85%), mean degree 8.8


wrote gowalla_final_checkins.csv.gz  (903,045 rows, 12.7 MB)
wrote gowalla_final_catalogue.csv  (81,240 rows, 5.1 MB)
wrote gowalla_final_users.csv  (31,667 rows, 0.5 MB)


wrote gowalla_final_friendships.csv  (235,898 rows, 4.2 MB)
wrote gowalla_final_stats.csv  (9 rows, 0.0 MB)


## 6. Caveats, for the write-up

- **The `>=10` sentence provably does not describe this column.** Their mean (27.16 ck/user) is
  *below* the unfiltered in-region mean at every region tried; activity cuts only raise it. On
  Foursquare the same sentence at least produced a 0.938 build — here it lands at 0.74 with
  0.37× their users. Their Foursquare/Weeplace tables are inherited from Long et al. WWW'24
  ([28]); no earlier paper carries this Gowalla column, so it is likely their own pipeline,
  undocumented.
- **`#categories` carries a hard bound**: 537 exceeds every in-region basis this dump can
  produce (max 344) and undershoots every whole-history basis (637–667). Either their category
  data is not this dump's `spot_categories`, or the cell was computed at a stage nobody reports.
- **The crawl outlives SNAP's window** (Jan 2009 – Aug 2011 vs Feb 2009 – Oct 2010) and holds
  36.0M check-ins vs SNAP's 6.4M; SNAP-windowed variants were swept and always score worse.
- **R = 15 / R = 28 use Yang's Foursquare city centres** — their pipeline's centres are unknown;
  the 0.997 user match should be read as "a ~15 km metro radius", not as those coordinates.
- The friendship file is a **single crawl snapshot** — no `friendship_old`/`new` split exists, so
  the leakage rule from the Foursquare track cannot be applied here; the whole graph is tagged
  `crawl` and any temporal claim about edges is unsupported.